# Notebook 2 — Transform Silver Layer
**BankGuard | Microsoft Fabric**

## What does this notebook do?
We take the raw Bronze tables and **clean them up**. This is the Silver layer.

### What we fix here:
- ✅ Fix wrong data types (e.g., dates stored as strings)
- ✅ Remove duplicate rows
- ✅ Add useful calculated columns (age, transaction hour, NPA classification)
- ✅ Mask sensitive PII columns (email, phone)
- ✅ Handle null/missing values

**Run Notebook 1 before this one.**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType

print('Reading Bronze tables...')

# Read from the Bronze tables we created in Notebook 1
bronze_txn   = spark.read.table('bronze_transactions')
bronze_cust  = spark.read.table('bronze_customers')
bronze_loans = spark.read.table('bronze_loans')

print(f'Transactions : {bronze_txn.count():,}')
print(f'Customers    : {bronze_cust.count():,}')
print(f'Loans        : {bronze_loans.count():,}')

## Silver Customers

Key transformations:
- Parse date_of_birth properly
- Calculate customer age
- Mask email and phone (privacy)
- Remove duplicates by customer_id

In [ ]:
# ── Step 1: Fix date types ─────────────────────────────────────────────────
# In CSVs, dates often come in as plain strings like '1985-06-15'
# We need to convert them to proper Date type so we can do calculations

df_cust = bronze_cust.withColumn(
    'date_of_birth',
    F.to_date(F.col('date_of_birth'), 'yyyy-MM-dd')   # parse string → date
).withColumn(
    'relationship_start_dt',
    F.to_date(F.col('relationship_start_dt'), 'yyyy-MM-dd')
)

# ── Step 2: Add calculated columns ────────────────────────────────────────
df_cust = df_cust.withColumns({

    # Calculate age from date of birth
    # datediff gives days difference, divide by 365 to get years
    'age': F.floor(
        F.datediff(F.current_date(), F.col('date_of_birth')) / 365
    ).cast(IntegerType()),

    # Group customers into age buckets — useful for reports
    'age_group': F.when(F.col('age') < 30, 'Under 30')
                  .when(F.col('age') < 45, '30–44')
                  .when(F.col('age') < 60, '45–59')
                  .otherwise('60+'),

    # How many years has this customer been with the bank?
    'years_with_bank': F.round(
        F.datediff(F.current_date(), F.col('relationship_start_dt')) / 365, 1
    ),

    # Is KYC complete? Simple True/False flag
    'is_kyc_done': F.col('kyc_status') == 'VERIFIED',

    # Mask email for privacy: john.doe@gmail.com → j***@gmail.com
    # We keep the first letter and domain, hide the rest
    'email_masked': F.regexp_replace(
        F.col('email'),
        r'(?<=.).(?=[^@]*@)',   # regex: replace chars after 1st, before @
        '*'
    ),

    # Mask phone: show only last 4 digits
    'phone_masked': F.concat(
        F.lit('XXXXXX'),
        F.substring(F.col('phone'), -4, 4)   # last 4 chars
    ),
})

# ── Step 3: Remove duplicate customers ────────────────────────────────────
# dropDuplicates keeps only one row per customer_id
before = df_cust.count()
df_cust = df_cust.dropDuplicates(['customer_id'])
after  = df_cust.count()
print(f'Duplicates removed: {before - after}')

# ── Step 4: Drop original PII columns, keep only masked versions ──────────
df_cust_silver = df_cust.drop('email', 'phone', 'pan_number')

# Save to Silver
df_cust_silver.write.format('delta').mode('overwrite').saveAsTable('silver_customers')
print(f'✅ silver_customers: {df_cust_silver.count():,} rows')

# Preview the result
df_cust_silver.select('customer_id','age','age_group','years_with_bank','is_kyc_done').show(5)

## Silver Transactions

Key transformations:
- Parse transaction timestamp
- Extract hour, day of week from timestamp
- Add amount buckets
- Flag international transactions
- Remove negative amounts (invalid data)

In [ ]:
# ── Step 1: Fix data types ─────────────────────────────────────────────────
df_txn = bronze_txn.withColumns({
    'transaction_ts' : F.to_timestamp(F.col('transaction_ts'), 'yyyy-MM-dd HH:mm:ss'),
    'amount'         : F.col('amount').cast(DecimalType(18, 2)),
})

# ── Step 2: Remove invalid rows ────────────────────────────────────────────
# Transactions with zero or negative amounts are data errors — remove them
invalid_count = df_txn.filter(F.col('amount') <= 0).count()
df_txn = df_txn.filter(F.col('amount') > 0)
print(f'Invalid transactions removed: {invalid_count}')

# ── Step 3: Add calculated columns ────────────────────────────────────────
df_txn = df_txn.withColumns({

    # Extract just the date part (for daily aggregations)
    'transaction_date': F.to_date(F.col('transaction_ts')),

    # Extract year-month for monthly reports (e.g. '2024-01')
    'year_month': F.date_format(F.col('transaction_ts'), 'yyyy-MM'),

    # What hour did the transaction happen? (0–23)
    'txn_hour': F.hour(F.col('transaction_ts')),

    # What day of the week? (1=Sunday, 7=Saturday in Spark)
    'day_of_week': F.dayofweek(F.col('transaction_ts')),

    # Is it a weekend?
    'is_weekend': F.when(
        F.dayofweek(F.col('transaction_ts')).isin([1, 7]), True
    ).otherwise(False),

    # Is the merchant in another country?
    'is_international': F.when(
        F.col('merchant_country') != 'IN', True
    ).otherwise(False),

    # Group transaction amounts into buckets for easy filtering
    # These thresholds are in Indian Rupees (INR)
    'amount_category': F.when(F.col('amount') < 1000,   'Small (<1K)')
                        .when(F.col('amount') < 10000,  'Medium (1K–10K)')
                        .when(F.col('amount') < 100000, 'Large (10K–1L)')
                        .otherwise('Very Large (>1L)'),

    # Time of day segment
    'time_of_day': F.when(F.hour('transaction_ts').between(6, 11),  'Morning')
                    .when(F.hour('transaction_ts').between(12, 17), 'Afternoon')
                    .when(F.hour('transaction_ts').between(18, 21), 'Evening')
                    .otherwise('Night'),
})

# ── Step 4: Remove duplicates ──────────────────────────────────────────────
before = df_txn.count()
df_txn_silver = df_txn.dropDuplicates(['transaction_id'])
print(f'Duplicate transactions removed: {before - df_txn_silver.count()}')

# Save to Silver
df_txn_silver.write.format('delta').mode('overwrite').saveAsTable('silver_transactions')
print(f'✅ silver_transactions: {df_txn_silver.count():,} rows')

# Preview
df_txn_silver.select(
    'transaction_id','amount','transaction_date','txn_hour',
    'is_weekend','is_international','amount_category'
).show(5)

## Silver Loans

Key transformations:
- Parse dates
- Add NPA classification (RBI standard)
- Calculate provision requirement
- Add loan health flag

In [ ]:
# Fix date types
df_loans = bronze_loans.withColumns({
    'disbursement_date'  : F.to_date(F.col('disbursement_date'), 'yyyy-MM-dd'),
    'maturity_date'      : F.to_date(F.col('maturity_date'), 'yyyy-MM-dd'),
    'last_payment_date'  : F.to_date(F.col('last_payment_date'), 'yyyy-MM-dd'),
    'principal_amount'   : F.col('principal_amount').cast(DecimalType(18, 2)),
    'outstanding_amount' : F.col('outstanding_amount').cast(DecimalType(18, 2)),
    'emi_amount'         : F.col('emi_amount').cast(DecimalType(18, 2)),
    'dpd'                : F.col('dpd').cast(IntegerType()),
})

# ── NPA Classification (RBI Guidelines) ───────────────────────────────────
# DPD = Days Past Due (how many days overdue the loan payment is)
# RBI has defined categories based on DPD

df_loans = df_loans.withColumns({

    # RBI NPA categories
    'npa_category': F.when(F.col('dpd') == 0,                'Standard')       # Paying on time
                    .when(F.col('dpd').between(1, 30),        'Special Mention') # 1–30 days late
                    .when(F.col('dpd').between(31, 90),       'Sub-Standard')    # 31–90 days late
                    .when(F.col('dpd').between(91, 180),      'Doubtful')        # 91–180 days late
                    .otherwise('Loss'),                                           # >180 days late

    # What % of the outstanding amount should be set aside as provision?
    'provision_pct': F.when(F.col('dpd') == 0,               0.004)  # 0.4% for standard
                     .when(F.col('dpd').between(1, 30),      0.05)   # 5%
                     .when(F.col('dpd').between(31, 90),     0.15)   # 15%
                     .when(F.col('dpd').between(91, 180),    0.40)   # 40%
                     .otherwise(1.00),                               # 100% for Loss

    # Simple flag: is this loan NPA or not?
    'is_npa': F.when(F.col('dpd') > 90, True).otherwise(False),

    # Loan to Value ratio (how much is still owed vs collateral)
    'ltv_ratio': F.when(
        F.col('collateral_value') > 0,
        F.round(F.col('outstanding_amount') / F.col('collateral_value') * 100, 1)
    ).otherwise(None),
})

# Calculate provision amount (outstanding × provision %)
df_loans_silver = df_loans.withColumn(
    'provision_amount',
    F.round(F.col('outstanding_amount') * F.col('provision_pct'), 2)
)

df_loans_silver.write.format('delta').mode('overwrite').saveAsTable('silver_loans')
print(f'✅ silver_loans: {df_loans_silver.count():,} rows')

# Quick NPA summary
print('\nNPA Category Distribution:')
df_loans_silver.groupBy('npa_category').count().orderBy('npa_category').show()

## Summary

Silver layer is done! We now have 3 clean tables:
- `silver_customers` — with age, masked PII, KYC flag
- `silver_transactions` — with date parts, amount categories, weekend flag
- `silver_loans` — with NPA classification, provision amounts

**Next: Run Notebook 3 to build Gold reporting tables.**

In [ ]:
# Final check
print('Silver Layer Summary')
print('=' * 30)
for table in ['silver_customers', 'silver_transactions', 'silver_loans']:
    count = spark.sql(f'SELECT COUNT(*) as n FROM {table}').collect()[0]['n']
    cols  = len(spark.read.table(table).columns)
    print(f'{table:30s}  rows: {count:,}  columns: {cols}')
print('\n✅ Run Notebook 3 next!')